In [1]:
import math
import pickle
import folium
import pickle
import numpy as np
import pandas as pd
import rustworkx as rx
import geopandas as gpd

from tqdm import tqdm

In [2]:
schools_df = pd.read_csv("data/ofsted_bradford_data.csv")
urn_name_map = schools_df.set_index("URN")["School name"].to_dict()

In [3]:
with open("transit_graph_.pkl", "rb") as f:
    G = pickle.load(f)

with open("node_to_idx_.pkl", "rb") as f:
    node_to_idx = pickle.load(f)

In [4]:
LON_SCALE_FACTOR = math.cos(math.radians(53.8))

In [9]:
origin_oa = "E00054185"	#"E00053802"# "E00054618" #"E00140251"
dest_urn = 147067#142825#141002#137277#130909

dest_idx = node_to_idx[dest_urn]
dest_node = G[dest_idx]

def gcs_manhattan_node(node):
    return abs(dest_node["lat"] - node["lat"]) + (abs(dest_node["lon"] - node["lon"]) * LON_SCALE_FACTOR)

path_indices = rx.astar_shortest_path(
    G,
    node=node_to_idx[origin_oa],
    goal_fn=lambda node: (node["name"] == dest_urn),
    edge_cost_fn=lambda edge: edge["weight"],
    estimate_cost_fn=gcs_manhattan_node,
)

optimal_path = [G.get_node_data(i)["name"] for i in path_indices]

optimal_duration = 0.0
for i in range(len(path_indices) - 1):
    edge_data = G.get_edge_data(path_indices[i], path_indices[i+1])
    optimal_duration += edge_data["weight"]

optimal_duration // 60

40.0

In [10]:
origin_idx = node_to_idx[origin_oa]
dest_idx = node_to_idx[dest_urn]

if G.has_edge(origin_idx, dest_idx):
    direct_edge = G.get_edge_data(origin_idx, dest_idx)
    direct_duration_mins = direct_edge["weight"] // 60
    print(f"Direct edge exists with duration: {direct_duration_mins} mins")
else:
    print("No direct edge exists between these nodes.")

Direct edge exists with duration: 61.0 mins


In [11]:
# Distinct color palette for transit legs
TRANSIT_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
current_color_idx = -1
is_in_transit = False

start_node = G.get_node_data(path_indices[0])
print(f"Start: ({start_node["lat"]}, {start_node["lon"]})")
dest_node = G.get_node_data(path_indices[-1])
print(f"Dest: ({dest_node["lat"]}, {dest_node["lon"]})")

# Draw map base
# m = folium.Map(location=[(start_node["lat"] + dest_node["lat"])/2, (start_node["lon"] + dest_node["lon"])/2], zoom_start=11, tiles="Cartodb Positron")
m = folium.Map(location=[(start_node["lat"] + dest_node["lat"])/2, (start_node["lon"] + dest_node["lon"])/2], zoom_start=11, tiles='https://basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png?key=cb1_33at_1_b265fe3928223dc30d291ece', attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>, &copy; <a href="https://carto.com/attributions">CARTO</a>')

# Draw origin
folium.Marker(
    location=(start_node["lat"], start_node["lon"]),
    popup=f"{start_node['name']}",
    icon=folium.Icon(color="green", icon="play")
).add_to(m)

# Draw destination
folium.Marker(
    location=(dest_node["lat"], dest_node["lon"]),
    popup=f"{urn_name_map[dest_node['name']]}",
    icon=folium.Icon(color="red", icon="flag")
).add_to(m)

# ---------------------------------------------------------
# Plot edges with state-based color swapping
# ---------------------------------------------------------
for u, v in zip(path_indices[:-1], path_indices[1:]):
    u_node = G.get_node_data(u)
    v_node = G.get_node_data(v)
    
    # Safely retrieve edge attributes
    edge_data = G.get_edge_data(u, v)
    
    # Handle NetworkX MultiGraph edge unwrapping if applicable
    if isinstance(edge_data, dict) and 0 in edge_data:
        edge_data = edge_data[0]
        
    edge_type = edge_data.get("type") if isinstance(edge_data, dict) else None

    segment_coords = [
        (u_node["lat"], u_node["lon"]),
        (v_node["lat"], v_node["lon"])
    ]

    if edge_type == "transit":
        # Entering a new bus leg after walking (or start of route)
        if not is_in_transit:
            is_in_transit = True
            current_color_idx = (current_color_idx + 1) % len(TRANSIT_COLORS)
            
        bus_color = TRANSIT_COLORS[current_color_idx]
        folium.PolyLine(
            locations=segment_coords,
            color=bus_color,
            weight=5,
            opacity=0.85,
            popup=f"Bus Leg ({edge_data.get('weight', 0):.1f} s)"
        ).add_to(m)

    else:
        # Any non-transit segment (walk, transfer, etc.) breaks the transit leg
        is_in_transit = False
        
        walk_weight = edge_data.get("weight", 0) / 60 if isinstance(edge_data, dict) else 0
        folium.PolyLine(
            locations=segment_coords,
            color="#555555",
            weight=3,
            dash_array="5, 8",
            opacity=0.8,
            popup=f"Walk ({walk_weight:.1f} mins)"
        ).add_to(m)

m.save(f"output/travel_map_{origin_oa}_{dest_urn}.html")

Start: (53.76332211347541, -1.748249354340917)
Dest: (53.7855637, -1.7455193)
